# Predicting Student Health Risk

Monitoring and modeling student health trajectories is essential for advancing early risk detection and preventive care. The dataset provides longitudinal observations encompassing lifestyle behaviors, physiological indicators, and psychological factors, offering a multidimensional foundation for analyzing determinants of well‑being in college populations. By examining temporal variation in these attributes, predictive modeling can contribute to a deeper understanding of health disparities, behavioral influences, and intervention effectiveness.

Yao Yan, Walter Reade, Elizabeth Park. Predicting Student Health Risk. https://kaggle.com/competitions/playground-series-s6e7, 2026. Kaggle.

## About the data

The data consists of `690088` student health records generated from the [College Student Health Behavior Dataset](https://www.kaggle.com/datasets/ziya07/college-student-health-behavior-dataset), which is modeled on large‑scale health studies of college populations.  Each record represents a time‑stamped observation describing lifestyle behaviors, physiological measurements, and psychological indicators. The dataset includes `13` feature columns and `1` target column (`health_condition`), which classifies each student as fit, at‑risk, or unhealthy.

| **Field** | **Description** |
| --- | --- |
| **[health_condition](ca://s?q=Explain_health_condition_label)** | Target label indicating overall health status (fit, at‑risk, unhealthy) |
| **[sleep_duration](ca://s?q=Explain_sleep_duration_feature)** | Total hours of sleep per day |
| **[heart_rate](ca://s?q=Explain_heart_rate_feature)** | Average resting heart rate (bpm) |
| **[bmi](ca://s?q=Explain_BMI_feature)** | Body Mass Index derived from height and weight |
| **[calorie_expenditure](ca://s?q=Explain_calorie_expenditure_feature)** | Estimated daily calories burned |
| **[step_count](ca://s?q=Explain_step_count_feature)** | Number of steps taken per day |
| **[exercise_duration](ca://s?q=Explain_exercise_duration_feature)** | Minutes spent exercising daily |
| **[water_intake](ca://s?q=Explain_water_intake_feature)** | Daily water consumption (liters) |
| **[diet_type](ca://s?q=Explain_diet_type_feature)** | Categorical diet classification (balanced, high‑carb, high‑fat, etc.) |
| **[stress_level](ca://s?q=Explain_stress_level_feature)** | Self‑reported stress score |
| **[sleep_quality](ca://s?q=Explain_sleep_quality_feature)** | Subjective sleep quality rating |
| **[physical_activity_level](ca://s?q=Explain_physical_activity_level_feature)** | Overall activity level category |
| **[smoking_alcohol](ca://s?q=Explain_smoking_alcohol_feature)** | Combined indicator of smoking and alcohol habits |
| **[gender](ca://s?q=Explain_gender_feature)** | Student gender category |

In [1]:
pip install optuna-integration[lightgbm] --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 7.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Modeling
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Metrics & splitting
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score

# Optuna
import optuna
from optuna.integration import LightGBMPruningCallback, XGBoostPruningCallback

# Jupyter cell magic
try:
    from IPython.core.magic import register_cell_magic
    @register_cell_magic
    def skip(line, cell): return
except:
    pass

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `optuna.integration.lightgbm` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.lightgbm` instead.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: `optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.
  return _bootstrap._gcd_import(name[level:], package, level)


In [3]:
season = "s6"
episode = "e7"

base_path = f"/kaggle/input/competitions/playground-series-{season}{episode}"

train = pd.read_csv(f"{base_path}/train.csv", index_col="id")
test = pd.read_csv(f"{base_path}/test.csv", index_col="id")
submission_sample = pd.read_csv(f"{base_path}/sample_submission.csv")

In [4]:
def downcasting(data: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    mem_before = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {mem_before:.2f} MB")
    
    for col in data.select_dtypes(include=["number"]).columns:
        if pd.api.types.is_integer_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="integer")
        elif pd.api.types.is_float_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="float")
    
    mem_after = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {mem_after:.2f} MB")
        print(f"Decreased by {(100 * (mem_before - mem_after) / mem_before):.1f}%\n")
    
    return data

print("Train train:")
train = downcasting(train)
print("Test train:")
test = downcasting(test)

Train train:
Memory usage of dataframe is 78.97 MB
Memory usage after optimization is: 60.55 MB
Decreased by 23.3%

Test train:
Memory usage of dataframe is 31.59 MB
Memory usage after optimization is: 23.69 MB
Decreased by 25.0%



In [5]:
target = "health_condition"

X = train.drop(target, axis=1)
y = train[target].astype("category").cat.codes

## Handling Missing Value

In [6]:
# Check total missing values per column
missing_counts = X.isnull().sum().sort_values(ascending=False)
print("Missing values per column:")
print(missing_counts)

# Percentage missing
missing_percent = (X.isnull().sum() / len(X)) * 100
print("\nMissing percentage per column:")
print(missing_percent)

Missing values per column:
stress_level               82811
sleep_duration             75999
sleep_quality              58331
calorie_expenditure        52853
water_intake               43477
physical_activity_level    36621
smoking_alcohol            28582
gender                     21373
step_count                 13916
bmi                        13898
heart_rate                  7833
diet_type                   6901
exercise_duration           6901
dtype: int64

Missing percentage per column:
sleep_duration             11.012943
heart_rate                  1.135073
bmi                         2.013946
calorie_expenditure         7.658878
step_count                  2.016554
exercise_duration           1.000017
water_intake                6.300211
diet_type                   1.000017
stress_level               12.000064
sleep_quality               8.452690
physical_activity_level     5.306715
smoking_alcohol             4.141791
gender                      3.097141
dtype: float64


In [7]:
# =========================================================
# Imputation strategy
# =========================================================

# Continuous → median
continuous_cols = [
    "sleep_duration", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake"
]
median_imputer = SimpleImputer(strategy="median")
X[continuous_cols] = median_imputer.fit_transform(X[continuous_cols])

# Ordinal categorical → mode (strings)
ordinal_cols = ["stress_level", "sleep_quality", "physical_activity_level"]
mode_imputer = SimpleImputer(strategy="most_frequent")
X[ordinal_cols] = mode_imputer.fit_transform(X[ordinal_cols])

# Nominal categorical → mode (strings)
categorical_cols = ["diet_type", "gender", "smoking_alcohol"]
X[categorical_cols] = mode_imputer.fit_transform(X[categorical_cols])

# =========================================================
# Verify missingness resolved
# =========================================================
print("\nRemaining missing values:", X.isnull().sum().sum())


Remaining missing values: 0


## Feature Engineering

Feature engineering was applied to convert raw behavioral and physiological measurements into more informative predictors of student health. I created composite indicators for sleep efficiency, activity intensity, hydration balance, and stress interactions, along with nonlinear BMI and heart‑rate transformations. Categorical attributes such as diet type and gender were encoded to ensure they could be effectively used by machine‑learning models.

In [8]:
def feature_engineer(df):
    df = df.copy()

    # =====================================================
    # 0. Encode categorical features IN-PLACE
    # =====================================================

    # Ordinal encodings
    sleep_quality_map = {"poor": 0, "average": 1, "good": 2}
    df["sleep_quality"] = df["sleep_quality"].map(sleep_quality_map).fillna(1)

    stress_map = {"low": 0, "medium": 1, "high": 2}
    df["stress_level"] = df["stress_level"].map(stress_map).fillna(1)

    activity_map = {"sedentary": 0, "moderate": 1, "active": 2}
    df["physical_activity_level"] = df["physical_activity_level"].map(activity_map).fillna(1)

    # Nominal encodings
    df["diet_type"] = df["diet_type"].astype("category").cat.codes
    df["gender"] = df["gender"].astype("category").cat.codes
    df["smoking_alcohol"] = df["smoking_alcohol"].astype("category").cat.codes

    # =====================================================
    # 1. Sleep-related features
    # =====================================================
    df["low_sleep"] = (df["sleep_duration"] < 6).astype(int)
    df["high_sleep"] = (df["sleep_duration"] > 9).astype(int)
    df["sleep_efficiency"] = df["sleep_quality"] / (df["sleep_duration"] + 1e-3)

    # =====================================================
    # 2. Activity & movement features
    # =====================================================
    df["steps_per_min_exercise"] = df["step_count"] / (df["exercise_duration"] + 1e-3)
    df["calories_per_step"] = df["calorie_expenditure"] / (df["step_count"] + 1e-3)
    df["activity_ratio"] = df["physical_activity_level"] / (df["exercise_duration"] + 1e-3)

    # =====================================================
    # 3. Hydration & diet interactions
    # =====================================================
    df["water_per_calorie"] = df["water_intake"] / (df["calorie_expenditure"] + 1e-3)
    df["water_per_step"] = df["water_intake"] / (df["step_count"] + 1e-3)

    # =====================================================
    # 4. Stress-related features
    # =====================================================
    df["stress_sleep_interaction"] = df["stress_level"] * df["sleep_duration"]
    df["stress_activity_interaction"] = df["stress_level"] * df["physical_activity_level"]
    df["high_stress"] = (df["stress_level"] > df["stress_level"].median()).astype(int)

    # =====================================================
    # 5. BMI transformations
    # =====================================================
    df["bmi_log"] = np.log1p(df["bmi"])
    df["bmi_squared"] = df["bmi"] ** 2
    df["is_obese"] = (df["bmi"] >= 30).astype(int)
    df["is_underweight"] = (df["bmi"] < 18.5).astype(int)

    # =====================================================
    # 6. Heart rate features
    # =====================================================
    df["hr_log"] = np.log1p(df["heart_rate"])
    df["hr_bmi_ratio"] = df["heart_rate"] / (df["bmi"] + 1e-3)

    # =====================================================
    # 7. Lifestyle composite features
    # =====================================================
    df["overall_wellness_score"] = (
        df["sleep_quality"] +
        df["physical_activity_level"] +
        df["water_intake"] -
        df["stress_level"]
    )

    df["risk_behavior_score"] = (
        df["smoking_alcohol"] +
        df["high_stress"] +
        df["low_sleep"]
    )

    return df

X = feature_engineer(X)
test = feature_engineer(test)

## Evaluation Metric: Balanced Accuracy

Balanced accuracy measures how well a classifier performs across all classes by giving each class equal weight, regardless of class imbalance. It is defined as the average recall across all \(K\) classes:

$\text{Balanced Accuracy} = \frac{1}{K} \sum_{i=1}^{K} \text{Recall}_i$

where

$\text{Recall}_i = \frac{\text{TP}_i}{\text{TP}_i + \text{FN}_i}.$

This metric is more effective than standard accuracy for imbalanced datasets because it prevents majority classes from dominating the score and ensures that minority classes contribute equally to the evaluation.

## Modelling

I selected XGBoost, LightGBM, and CatBoost because gradient‑boosted tree models are the strongest choice for structured tabular data, especially when nonlinear relationships and engineered interaction features are present. These methods handle mixed numeric and categorical inputs, are robust to outliers, and work well without feature scaling. Each model adds a different strength: XGBoost provides stability and strong regularization, LightGBM trains quickly and captures deep feature interactions, and CatBoost handles categorical structure effectively and produces well‑calibrated probabilities. Together, they form a reliable and complementary ensemble for this task.

In [9]:
# ---------------------------------------------------------
# Dataset split for tuning
# ---------------------------------------------------------
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
%%skip
def objective_xgb(trial):
    params = {
        "objective": "multi:softprob",
        "num_class": 3,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "verbosity": 0,
        "random_state": 42,

        # Core hyperparameters
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "max_depth": trial.suggest_int("max_depth", 4, 10),

        # Sampling
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        # Regularization
        "gamma": trial.suggest_float("gamma", 0, 3),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.5, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 2.0),
    }

    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid, preds)

    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=30)
best_xgb_params = study_xgb.best_params

In [11]:
%%skip
def objective_lgbm(trial):
    params = {
        "objective": "multiclass",
        "num_class": 3,
        "metric": "multi_logloss",
        "device": "cpu",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "seed": 42,

        # Core hyperparameters
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 12),

        # Regularization
        "lambda_l1": trial.suggest_float("lambda_l1", 0.0, 2.0),
        "lambda_l2": trial.suggest_float("lambda_l2", 0.0, 5.0),

        # Sampling
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
    }

    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid)

    model = lgb.train(
        params,
        train_data,
        valid_sets=[valid_data]
    )

    preds = model.predict(X_valid)
    preds_class = preds.argmax(axis=1)
    score = balanced_accuracy_score(y_valid, preds_class)

    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score


study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=30)
best_lgbm_params = study_lgbm.best_params

In [12]:
%%skip
def objective_catboost(trial):
    params = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "task_type": "CPU", 
        "random_seed": 42,
        "verbose": False,

        # Core hyperparameters
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.15),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),

        # Tree complexity
        "border_count": trial.suggest_int("border_count", 32, 255),
    }

    model = CatBoostClassifier(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_valid)
    score = balanced_accuracy_score(y_valid, preds)

    trial.report(score, step=0)
    if trial.should_prune():
        raise optuna.TrialPruned()

    return score


study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_catboost, n_trials=30)
best_cat_params = study_cat.best_params

In [13]:
best_xgb_params = {'n_estimators': 386, 'learning_rate': 0.09090259261685009, 'max_depth': 7, 'subsample': 0.7417278264708168, 'colsample_bytree': 0.8926859603796309, 'gamma': 0.4316850324344384, 'reg_lambda': 3.3143585419960533, 'reg_alpha': 1.4134717281933693}
best_lgb_params = {'learning_rate': 0.12994813996332122, 'num_leaves': 226, 'max_depth': -1, 'lambda_l1': 1.5353088579085583, 'lambda_l2': 0.046889704305499436, 'feature_fraction': 0.8010871699035738, 'bagging_fraction': 0.861711880697545, 'bagging_freq': 5}
best_cat_params = {'learning_rate': 0.11474830227432195, 'depth': 5, 'l2_leaf_reg': 7.237649694886767, 'border_count': 210}

In [19]:
xgb_final = xgb.XGBClassifier(**best_xgb_params, verbosity=0)
lgb_final = lgb.LGBMClassifier(**best_lgb_params, verbosity=-1)
cat_final  = CatBoostClassifier(**best_cat_params, verbose=False)

xgb_final.fit(X, y)
lgb_final.fit(X, y)
cat_final.fit(X, y)

xgb_p = xgb_final.predict_proba(test)
lgb_p = lgb_final.predict_proba(test)
cat_p = cat_final.predict_proba(test)

xgb_pred = xgb_p.argmax(axis=1)
lgb_pred = lgb_p.argmax(axis=1)
cat_pred = cat_p.argmax(axis=1)

In [23]:
%%skip
# ---------------------------------------------------------
# Tune blending weights with Optuna
# ---------------------------------------------------------
def objective_blend(trial):
    w_xgb = trial.suggest_float("w_xgb", 0.0, 1.0)
    w_lgb = trial.suggest_float("w_lgb", 0.0, 1.0)
    w_cat  = trial.suggest_float("w_cat",  0.0, 1.0)

    total = w_xgb + w_lgb + w_cat + 1e-9

    xgb_p = xgb_final.predict_proba(X_valid)
    lgb_p = lgb_final.predict_proba(X_valid)
    cat_p  = cat_final.predict_proba(X_valid)

    blend_p = (w_xgb*xgb_p + w_lgb*lgb_p + w_cat*cat_p) / total
    blend_pred = np.argmax(blend_p, axis=1)

    return balanced_accuracy_score(y_valid, blend_pred)

study_blend = optuna.create_study(direction="maximize")
study_blend.optimize(objective_blend, n_trials=30)
best_weights = study_blend.best_params

print("Best blending weights:", best_weights)

In [24]:
best_weights = {'w_xgb': 0.14426265132777288, 'w_lgb': 0.6264776330203634, 'w_cat': 0.012330193375863704}

In [25]:
# ---------------------------------------------------------
# Consensus logic
# ---------------------------------------------------------
final_pred = np.zeros(len(test), dtype=int)

for i in range(len(test)):
    preds = [xgb_pred[i], lgb_pred[i], cat_pred[i]]

    # ---------------------------------------------
    # Case 1: Unanimous agreement → trust consensus
    # ---------------------------------------------
    if preds.count(preds[0]) == 3:
        final_pred[i] = preds[0]
        continue

    # ---------------------------------------------
    # Case 2: Majority vote (2 out of 3 agree)
    # ---------------------------------------------
    if len(set(preds)) == 2:
        # majority class
        maj = max(set(preds), key=preds.count)
        final_pred[i] = maj
        continue

    # ---------------------------------------------
    # Case 3: Full disagreement → fallback blending
    # ---------------------------------------------
    # Weighted average of probabilities
    blend_p = (
        best_weights["w_xgb"] * xgb_p[i] +
        best_weights["w_lgb"] * lgb_p[i] +
        best_weights["w_cat"] * cat_p[i]
    )
    blend_p /= blend_p.sum()

    blend_pred = blend_p.argmax()

    # ---------------------------------------------
    # Case 4: If blending is indecisive → use highest-confidence model
    # ---------------------------------------------
    max_conf = max(
        xgb_p[i].max(),
        lgb_p[i].max(),
        cat_p[i].max()
    )

    if max_conf < 0.55:
        # fallback to highest-confidence model
        if xgb_p[i].max() == max_conf:
            final_pred[i] = xgb_pred[i]
        elif lgb_p[i].max() == max_conf:
            final_pred[i] = lgb_pred[i]
        else:
            final_pred[i] = cat_pred[i]
    else:
        final_pred[i] = blend_pred

In [26]:
categories = train[target].astype('category').cat.categories
final_labels = categories[final_pred]

In [27]:
submission = pd.DataFrame({
    'id': test.index, 
    'class': final_labels
})
submission.to_csv('submission.csv', index=False)